# To Do:
- add 'TO DO' throughout code comments

In [ ]:
# add these dataset locally from kaggle: 
# 'statcanfoodpricesfeb2022'
# 'su-eatable-life' - https://figshare.com/articles/dataset/SU-EATABLE_LIFE_a_comprehensive_database_of_carbon_and_water_footprints_of_food_commodities/13271111
# 'cosco-prices'

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Import Data

In [ ]:
df_macro = pd.read_csv("/kaggle/input/nutrition-details-for-most-common-foods/nutrients_csvfile.csv")
df_micro = pd.read_csv("/kaggle/input/food-nutrition-dataset/food.csv")
df_glob_cost = pd.read_csv("/kaggle/input/global-food-prices/wfp_market_food_prices.csv",encoding="ISO-8859-1")
df_can_cost = pd.read_csv("../input/statcanfoodpricesfeb2022/1810000201_databaseLoadingData.csv")
df_ghg = pd.read_csv("/kaggle/input/environment-impact-of-food-production/Food_Production.csv")
df_sueatable_ghg = pd.read_csv("/kaggle/input/su-eatable-life/CF_ITEMS.csv")
df_cosco_cost = pd.read_csv("/kaggle/input/cosco-prices/Cosco_Prices.csv")


print(f'Micronutrients:\t{df_macro.shape}')
print(f'Macronutrients:\t{df_micro.shape}')
print(f'Global Costs:\t{df_glob_cost.shape}')
print(f'Canadian Costs:\t{df_can_cost.shape}')
print(f'Cosco Costs:\t{df_cosco_cost.shape}')
print(f'Emissions:\t{df_ghg.shape}')

# Data Cleansing

## Macro

In [ ]:
print(df_macro.columns)
print('\n',pd.unique(df_macro['Category'])) # average and use these categories instead?
df_macro.head()

In [ ]:
if 'abc' in ['abc','asdf']:
    print(True)

In [ ]:
df_macro_1 = df_macro.copy() # reset operations
df_macro_1 = df_macro_1.drop(columns=['Sat.Fat','Fiber','Measure'])
df_macro_1['Category'] = df_macro_1['Category'].apply(lambda x: 'Vegetables' if x in ['Vegetables A-E','Vegetables F-P','Vegetables R-Z'] else (\
                                                                'Fruits' if x in ['Fruits A-F','Fruits G-P','Fruits R-Z'] else (x)))
df_macro_1.drop_duplicates(subset = ['Food'], keep = 'first', inplace = True)
df_macro_1 = df_macro_1.rename(columns=str.lower)
df_macro_1 = df_macro_1.add_prefix('macro_')
df_macro_1['macro_food'] = df_macro_1['macro_food'].str.lower() # lower case primary key for inner join

# Normalize Columns
df_macro_1 = df_macro_1.replace(',','', regex=True)
# df_macro_1[df_macro_1['macro_calories'].isnull()] # find problem rows
df_macro_1 = df_macro_1.drop([91,134,205]).reset_index() # drop problem rows
df_macro_1['macro_protein'],df_macro_1['macro_fat'],df_macro_1['macro_carbs']\
    = df_macro_1['macro_protein'].replace('t','0'),df_macro_1['macro_fat'].replace('t','0'),df_macro_1['macro_carbs'].replace('t','0')

# Per kg measures
df_macro_1['macro_calories'] = df_macro_1['macro_calories'].astype(float).divide(df_macro_1['macro_grams'].astype(float)).multiply(1000).astype(int)
df_macro_1['macro_protein'] = df_macro_1['macro_protein'].astype(float).divide(df_macro_1['macro_grams'].astype(float)).multiply(1000).astype(int)
df_macro_1['macro_fat'] = df_macro_1['macro_fat'].astype(float).divide(df_macro_1['macro_grams'].astype(float)).multiply(1000).astype(int)
df_macro_1['macro_carbs'] = df_macro_1['macro_carbs'].astype(float).divide(df_macro_1['macro_grams'].astype(float)).multiply(1000).astype(int)

df_macro_1 = df_macro_1.drop(columns=['index','macro_grams'])

df_macro_1


In [ ]:
print(pd.unique(df_macro_1['macro_food']).shape)
print(pd.unique(df_macro_1['macro_food']))

## Micro

In [ ]:
# print(df_micro.columns)
# print('\n',pd.unique(df_micro['Category']).shape)
# df_micro.head()

In [ ]:
# df_micro_1 = df_micro.copy()
# df_micro_1.insert(0 , 'food', df_micro_1['Category'].str.lower()) # lower case primary key for inner join
# df_micro_1.drop_duplicates(subset = ['food'], keep = 'first', inplace = True) # look into which duplicats to keep
# df_micro_1 = df_micro_1.drop(columns = ['Description','Nutrient Data Bank Number','Category']) 
# df_micro_1 = df_micro_1.rename(columns = str.lower)
# df_micro_1 = df_micro_1.add_prefix('micro_')
# df_micro_1.head()

In [ ]:
# print(pd.unique(df_micro_1['micro_food']).shape)
# print(pd.unique(df_micro_1['micro_food']))
# print(pd.unique(df_micro_1['micro_food'])[:100])

## Cost - Dataset 1

In [ ]:
# df_glob_cost

In [ ]:
# df_cost_1 = df_glob_cost.copy()
# df_cost_1 = df_cost_1.loc[df_cost_1['cur_name'] == 'USD'][['cm_name','mp_price','um_name']]
# df_cost_1.rename(columns = {'cm_name':'item', 'mp_price':'price', 'um_name':'measure'}, inplace = True)
# df_cost_1.add_prefix('cost_')
# pd.unique(df_cost_1['item'])

No CAD prices, poor items, try other dataset.  
From Statistics Canada:
https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810000201&cubeT

## Cost - Dataset 2 (stat can)

In [ ]:
# print(pd.unique(df_can_cost['Products']))
# df_can_cost".tail()

In [ ]:
# df_cost_2 = df_can_cost.copy()
# df_cost_2[['food','drop_split','measure']] = df_cost_2['Products'].str.split(',',expand=True)
# df_cost_2['food']
# df_cost_2.measure.fillna(df_cost_2.drop_split, inplace=True)
# df_cost_2.drop_duplicates(subset = ['food'], keep = 'first', inplace = True) # look into which duplicats to keep
# df_cost_2 = df_cost_2[['food','measure','VALUE']]
# df_cost_2['food'] = df_cost_2['food'].str.lower() # lower case primary key for inner join
# df_cost_2 = df_cost_2.rename(columns = str.lower)
# df_cost_2 = df_cost_2.add_prefix('cost_')
# df_cost_2.head()

In [ ]:
# print(pd.unique(df_cost_2['cost_food']).shape)
# print(pd.unique(df_cost_2['cost_food']))

## Cost - Dataset 3 (cosco)


In [ ]:
print(pd.unique(df_cosco_cost['Category']))
df_cosco_cost[df_cosco_cost['Category']=='Grocery']

In [ ]:
df_cost_3 = df_cosco_cost.copy()
# df_cost_3 = df_cost_3[['Product','Size','Price']]
df_cost_3.rename(columns={'Product':'food','Size':'size','Price':'price'}, inplace=True)
df_cost_3['food'] = df_cost_3['food'].str.lower() # lower case primary key for inner join
df_cost_3 = df_cost_3.add_prefix('cost_')

df_cost_3[['size_value', 'size_unit']] = df_cost_3['cost_size'].str.split(' ', 1, expand=True)
df_cost_3 = df_cost_3.drop(columns='cost_size')
print(pd.unique(df_cost_3['size_unit']))

df_cost_3[df_cost_3['size_value']=='1/2'] # find problem rows
df_cost_3 = df_cost_3.drop([194,195]).reset_index(drop=True) # drop problem rows

# # df_cost_3[df_cost_3['size_unit']=='ct']
df_cost_3

In [ ]:
# Get count conversions?

In [ ]:
# no general measures for 'ct' (individual item counts)
df_kg_conversions = pd.DataFrame(data={'to_kg':[0.0283495,\
                                          0.453592, 0.453592,\
                                          1, 1, 1, 0.001, \
                                          0.26, 0.26, 0.26, \
                                          0.9464, \
                                          0.001], \
                                       'standard_unit':['g',\
                                        'g','g',\
                                        'ml','ml','ml','ml',\
                                        'ml','ml','ml',\
                                        'ml',\
                                        'g']},\
                                  index = ['oz', \
                                        'lb', 'lbs', \
                                        'liter', 'liters',  'l', 'ml', \
                                        'gallon', 'gallons', 'gal', \
                                        'qts', \
                                        'g'])
df_kg_conversions

In [ ]:
# Inner Join Conversion Table
df_kg_cost = df_cost_3.merge(df_kg_conversions, left_on = 'size_unit', right_index=True, how = 'inner').reset_index(drop=True)

# Convert everything to kg (assume volume measures have density of water)
df_kg_cost['kg_size_value'] = df_kg_cost['size_value'].astype(float).multiply(df_kg_cost['to_kg'])
df_kg_cost['cost_price'] = df_kg_cost['cost_price'].astype(float).divide(df_kg_cost['kg_size_value']).round(2)

# TO DO : Use quantity (2 @ 100ml) instead
# Add standard unit for UI
df_kg_cost['unit'] = df_kg_cost['kg_size_value'].astype(float).multiply(1000).astype(int)
df_kg_cost['unit'] = df_kg_cost['unit'].apply(lambda x: 10 if x <= 10 else (\
                                                        50 if x <= 50 else (\
                                                        100 if x <= 100 else (\
                                                        250 if x <= 250 else (\
                                                        500 if x <= 500 else (\
                                                        1000 if x <= 1000 else (\
                                                        2000 if x <= 2000 else (5000))))))))
df_kg_cost['unit'] = df_kg_cost['unit'].astype(str) + df_kg_cost['standard_unit']
df_kg_cost[df_kg_cost['cost_food'].str.contains('orange')]

In [ ]:
df_kg_cost = df_kg_cost[['cost_food','cost_price', 'unit']]
df_kg_cost.sort_values(by='cost_price', ascending=False)

In [ ]:
print(pd.unique(df_cost_3['cost_food']).shape)
print(pd.unique(df_cost_3['cost_food']))

## Emissions - Dataset 1

In [ ]:
# print(df_ghg.columns)
# df_ghg.tail()

In [ ]:
# df_ghg_1 = df_ghg.copy()
# df_ghg_1.rename(columns = {'Total_emissions':'Emissions per kg', 'Food product':'food'}, inplace = True)
# df_ghg_1 = df_ghg_1[['food','Emissions per kg']]
# df_ghg_1 = df_ghg_1.rename(columns = str.lower)
# df_ghg_1['food'] = df_ghg_1['food'].str.lower() # lower case primary key for inner join
# df_ghg_1 = df_ghg_1.add_prefix('ghg_')
# df_ghg_1.head()

In [ ]:
# print(pd.unique(df_ghg_1['ghg_food']).shape)
# print(pd.unique(df_ghg_1['ghg_food']))

## Emissions - Dataset 2 (df_sueatable_ghg)

In [ ]:
print(df_sueatable_ghg.columns)
df_sueatable_ghg.tail()

In [ ]:
df_ghg_2 = df_sueatable_ghg.copy()
df_ghg_2 = df_ghg_2[['FOOD COMMODITY ITEM - kg CO2 eq/ kg or litre food commodity\t\t\t\t\t\t\t\t',\
                     'mean']]
df_ghg_2.rename(columns = {'FOOD COMMODITY ITEM - kg CO2 eq/ kg or litre food commodity\t\t\t\t\t\t\t\t':'food',\
                           'mean':'emissions'}, inplace = True)
df_ghg_2['food'] = df_ghg_2['food'].str.lower() # lower case primary key for inner join
df_ghg_2 = df_ghg_2.add_prefix('ghg_')
df_ghg_2.head()

In [ ]:
print(pd.unique(df_ghg_2['ghg_food']).shape)
print(pd.unique(df_ghg_2['ghg_food']))

# Full Join on Substrings

In [ ]:
print(df_ghg_2.loc[df_ghg_2['ghg_food'].str.contains("bread", case=False)]['ghg_food'])
print(df_kg_cost.loc[df_kg_cost['cost_food'].str.contains("bread", case=False)]['cost_food'])
print(df_macro_1.loc[df_macro_1['macro_food'].str.contains("bread", case=False)]['macro_food'])

# split on spaces

In [ ]:
# try join if primary_key_1 is sub string of primary_key_2

dataFrame1 = df_ghg_2.copy()
dataFrame2 = df_macro_1.copy()

dataFrame1['join'] = 1
dataFrame2['join'] = 1
  
dataFrameFull = dataFrame1.merge(
  dataFrame2, on='join').drop('join', axis=1)
  
dataFrame2.drop('join', axis=1, inplace=True)

dataFrameFull['match'] = dataFrameFull.apply(
    lambda x: x.macro_food.find(x.ghg_food), axis=1).ge(0)
  
df_substring_ghg_macro = dataFrameFull[dataFrameFull['match']]
df_substring_ghg_macro = df_substring_ghg_macro.drop(columns=['match'])

In [ ]:
print(len(df_substring_ghg_macro))
df_substring_ghg_macro.head()

df_substring_ghg_macro.drop_duplicates(subset = ['macro_food'], keep = 'first', inplace = True) 
df_substring_ghg_macro

In [ ]:
dataFrame1 = df_substring_ghg_macro.copy()
dataFrame2 = df_kg_cost.copy()

dataFrame1['join'] = 1
dataFrame2['join'] = 1
  
dataFrameFull = dataFrame1.merge(
  dataFrame2, on='join').drop('join', axis=1)
  
dataFrame2.drop('join', axis=1, inplace=True)

dataFrameFull['match'] = dataFrameFull.apply(
    lambda x: x.cost_food.find(x.ghg_food), axis=1).ge(0)
#     lambda x: x.ghg_food.find(x.cost_food), axis=1).ge(0) og
#     lambda x: x.macro_food.find(x.cost_food), axis=1).ge(0) 4
#     lambda x: x.cost_food.find(x.ghg_food), axis=1).ge(0) 56
#     lambda x: x.cost_food.find(x.macro_food), axis=1).ge(0) 47



df_substring_cost_ghg_macro = dataFrameFull[dataFrameFull['match']]
df_substring_cost_ghg_macro = df_substring_cost_ghg_macro.drop(columns=['match'])

In [ ]:
print(len(df_substring_cost_ghg_macro))
food_subset = 'macro_food' # select column with most generalized names
df_substring_cost_ghg_macro.drop_duplicates(subset = [food_subset], keep = 'first', inplace = True) 
print(len(df_substring_cost_ghg_macro))
print(pd.unique(df_substring_cost_ghg_macro['macro_food']))
df_substring_cost_ghg_macro.head()



In [ ]:
final_df = df_substring_cost_ghg_macro.copy()

final_df = final_df.reset_index()
final_df = final_df[['macro_food', 'macro_category', 'unit', 'ghg_emissions', 'macro_calories', 'macro_protein', 'macro_fat', 'macro_carbs', 'cost_price']]

# food_column = final_df.pop('macro_food')
# final_df.insert(0, 'food', food_column)

final_df = final_df.rename(columns={'macro_food':'food',\
                                  'macro_category':'category',\
                                  'macro_calories':'calories',\
                                  'macro_protein':'protein',\
                                  'macro_fat':'fat',\
                                  'macro_carbs':'carbs',\
                                  'ghg_emissions':'emissions',\
                                  'cost_price':'price'}) 
final_df.index.name = 'index'
final_df = final_df.sort_values(by='emissions').groupby(by='category').apply(lambda x: x).reset_index(drop=True)

final_df.to_csv('FoodPrint_DataSet.csv',index=True)
final_df

# Inner Join Datasets

In [ ]:
# # merge(left_df, right_df, on=’Customer_id’, how=’inner’)
# # df1.merge(df2,on='name').merge(df3,on='name')
# df_nutr = df_micro_1.merge(df_macro_1, left_on = 'micro_food', right_on = 'macro_food', how = 'inner')
# df_nutr_cost = df_nutr.merge(df_cost_2, left_on = 'micro_food', right_on = 'cost_food', how = 'inner')
# df_nutr_cost_ghg = df_nutr_cost.merge(df_ghg_1, left_on = 'micro_food', right_on = 'ghg_food', how = 'inner')
# df_cost_ghg = df_cost_2.merge(df_ghg_1, left_on = 'cost_food', right_on = 'ghg_food', how = 'inner')

## Join on sub strings of sub strings
'Chocolate Milk' should join with 'Chocolate' and 'Milk'

## Use NLP models or String Comparison techniques to join

In [ ]:
import gensim
model = gensim.models.Word2Vec.load_word2vec_format('path-to-vectors.txt', binary=False)
# if you vector file is in binary format, change to binary=True
sentence = ["London", "is", "the", "capital", "of", "Great", "Britain"]
vectors = [model[w] for w in sentence]

https://mccormickml.com/2016/04/12/googles-pretrained-word2vec-model-in-python/
https://code.google.com/archive/p/word2vec/
    

In [ ]:
http://aishelf.org/str-comparison/
https://towardsdatascience.com/string-comparison-is-easy-with-fuzzywuzzy-library-611cc1888d97
        